# **CODE DATASET**

For this project, we will be using the [CodeSearchNet](https://huggingface.co/datasets/claudios/code_search_net) dataset as our base, which contains a large collection of code snippets and their corresponding natural language descriptions. However, we will be fine-tuning our model on a smaller, more specific dataset that we have curated ourselves.

## **Exploring the Dataset**

First, let's login to Hugging Face to be able to push our dataset to the Hugging Face Hub.

In [1]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Now, we can load the main dataset and explore its structure.

In [2]:
from datasets import load_dataset

In [3]:
# Let's see an example of how the dataset looks like for one of the languages
dataset = load_dataset("claudios/code_search_net", "python")

README.md:   0%|          | 0.00/13.6k [00:00<?, ?B/s]

python/train-00000-of-00003.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

python/train-00001-of-00003.parquet:   0%|          | 0.00/135M [00:00<?, ?B/s]

python/train-00002-of-00003.parquet:   0%|          | 0.00/125M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/21.7M [00:00<?, ?B/s]

python/validation-00000-of-00001.parquet:   0%|          | 0.00/23.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

In [4]:
# Let's see the first example of the training set
dataset["train"][0]

{'repository_name': 'proycon/pynlpl',
 'func_path_in_repository': 'pynlpl/formats/folia.py',
 'func_name': 'AbstractElement.addidsuffix',
 'whole_func_string': 'def addidsuffix(self, idsuffix, recursive = True):\n        """Appends a suffix to this element\'s ID, and optionally to all child IDs as well. There is sually no need to call this directly, invoked implicitly by :meth:`copy`"""\n        if self.id: self.id += idsuffix\n        if recursive:\n            for e in self:\n                try:\n                    e.addidsuffix(idsuffix, recursive)\n                except Exception:\n                    pass',
 'language': 'python',
 'func_code_string': 'def addidsuffix(self, idsuffix, recursive = True):\n        """Appends a suffix to this element\'s ID, and optionally to all child IDs as well. There is sually no need to call this directly, invoked implicitly by :meth:`copy`"""\n        if self.id: self.id += idsuffix\n        if recursive:\n            for e in self:\n          

In [5]:
# Let's see the whole function string of the first example
print(dataset["train"][0]['whole_func_string'])

def addidsuffix(self, idsuffix, recursive = True):
        """Appends a suffix to this element's ID, and optionally to all child IDs as well. There is sually no need to call this directly, invoked implicitly by :meth:`copy`"""
        if self.id: self.id += idsuffix
        if recursive:
            for e in self:
                try:
                    e.addidsuffix(idsuffix, recursive)
                except Exception:
                    pass


In [6]:
# Let's see the documentation string of the first example
print(dataset["train"][0]['func_documentation_string'])

Appends a suffix to this element's ID, and optionally to all child IDs as well. There is sually no need to call this directly, invoked implicitly by :meth:`copy`


Considerations for this dataset:
- We need to ensure that the dataset is balanced across different programming languages to avoid bias towards any particular language.
- We need to make pairs of code snippets and their corresponding documentation, ensuring that they are correctly aligned.
- For code generation, the model should be able to generate code from the first line of code, half of a function, or a random point within it.

In [7]:
import random 

def create_pairs(code_snippet, language):
    # Extract the lines of code and the documentation string
    lines = code_snippet['whole_func_string'].split('\n')
    documentation = code_snippet['func_documentation_string']
    pairs = []
    
    # FISRT TASK: CODE COMPLETION
    
    # Fisrt type of pair: The first line of the code
    body = '\n'.join(lines[1:])
    if len(body.strip()) >= 20:
        pairs.append({
            'input_code': f'complete {language}: {lines[0]}',
            'target_completion': body,
            'task': 'completion',
            'lang': language
        })
    
    # Second type of pair: The half of the code
    mid  = len(lines) // 2
    top  = '\n'.join(lines[:mid])
    rest = '\n'.join(lines[mid:])
    
    if len(rest.strip()) >= 20:
        pairs.append({
            'input_code': f'complete {language}: {top}',
            'target_completion': rest,
            'task': 'completion',
            'lang': language
        })
        
    # Third type of completion: A random portion of the code
    if len(lines) >= 6:
        cut    = random.randint(1, len(lines) - 2)
        top_r  = '\n'.join(lines[:cut])
        rest_r = '\n'.join(lines[cut:])
        if len(rest_r.strip()) >= 20:
            pairs.append({
                'input_code': f'complete {language}: ' + top_r,
                'target_completion': rest_r,
                'task': 'completion',
                'lang': language
            })
            
    # SECOND TASK: DOCSTRING GENERATION
    if documentation and len(documentation.strip()) >= 10:
        pairs.append({
            'input_code': f'generate docstring {language}: ' + code_snippet['whole_func_string'],
            'target_completion': documentation.strip(),
            'task': 'docstring',
            'lang': language
        })

    return pairs

Import functions to create datasets and upload them to Hugging Face.

In [8]:
from datasets import DatasetDict, Dataset

Set a seed for reproducibility

In [9]:
random.seed(42)

Let's create the function to automate the process of loading different languages, creating pairs of code snippets and documentation, and creating the dataset.

In [10]:
def create_dataset(n_per_lenguages: int, languages: list[str]):
    all_pairs = []
    
    # Iterate over the specified languages and create pairs for each code snippet
    for lang in languages:
        ds = load_dataset("claudios/code_search_net", lang)
        subset = ds['train'].shuffle(seed=42).select(range(n_per_lenguages))
        for example in subset:
            all_pairs.extend(create_pairs(example, lang))
        
    # Convert to Hugging Face Dataset and shuffle
    full_ds = Dataset.from_list(all_pairs).shuffle(seed=42)
    
    # Calculate the number of examples for train, validation, and test sets
    total   = len(full_ds)
    n_val   = int(total * 0.10)
    n_test  = int(total * 0.05)
    n_train = total - n_val - n_test
    
    # Split into train, validation, and test sets
    dataset_dict = DatasetDict({
        "train":      full_ds.select(range(n_train)),
        "validation": full_ds.select(range(n_train, n_train + n_val)),
        "test":       full_ds.select(range(n_train + n_val, total))
    })
    
    # return the dataset dictionary
    return dataset_dict

To save our work, we will push the dataset to Hugging Face Hub and make a backup on Google Drive.

In [11]:
# Mount Google Drive to save the dataset there as well
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# Define the languages
languages  = ["python", "javascript", "java", "go", "ruby", "php"]

For this project, we will compare the training and the performance of two models: T5-large and an SLM-based model. We will fine-tune both models on the same dataset and evaluate their performance on a test set.

For the SLM-based model, we will use QLoRA to fine-tune it on our dataset. For that reasosn, the SLM-based model will be trained on a bigger dataset than the T5-large model because the quantization allows us to train on a larger dataset without running into memory issues.

In [13]:
models = {
    "T5": 1500, # 1500 examples per language for T5 model
    "SLM": 3000 # 3000 examples per language for the SLM-based model
}

In [14]:
# Iterate over the models and create, save, and push the dataset for each model
for model, number in models.items():
    print('='*50)
    print('[INFO] Creating dataset for', model)
    
    dataset = create_dataset(
        n_per_lenguages= number,
        languages= languages
    )
    
    name_hf = f"Juanxxo/smallcoder-{model.lower()}-dataset"
    
    save = dataset.save_to_disk(f"/content/drive/MyDrive/smallcoder/{model}/dataset")
    dataset.push_to_hub(name_hf, private=False)
    
    if save is None:
        print(f"[INFO] Dataset for {model} saved successfully.")
        print('Information about the dataset for', model)
        print(f"Train:      {len(dataset['train'])}")
        print(f"Validation: {len(dataset['validation'])}")
        print(f"Test:       {len(dataset['test'])}")
        print(f"Hugging Face Link: {name_hf}")
    else:
        print(f"[ERROR] Failed to save dataset for {model}.")

[INFO] Creating dataset for T5


javascript/train-00000-of-00001.parquet:   0%|          | 0.00/108M [00:00<?, ?B/s]

javascript/test-00000-of-00001.parquet:   0%|          | 0.00/5.46M [00:00<?, ?B/s]

javascript/validation-00000-of-00001.par(…):   0%|          | 0.00/6.98M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/123889 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  76%|#######6  | 1.05MB / 1.38MB            

[INFO] Dataset for SLM saved successfully.
Information about the dataset for SLM
Train:      57272
Validation: 6737
Test:       3368
Hugging Face Link: Juanxxo/smallcoder-slm-dataset
